# Data Science & Database Assignment: Session 13
## Topic: Window Functions - Part 1 (`ROW_NUMBER`, `RANK`, `DENSE_RANK`, and `PARTITION BY`)

**Student Name:** Atharv Chaudhari  
**Course:** Data Science & Database Fundamentals  
**Database Name:** `spotify_windows.db`  
**Core Concepts:** Window Functions, `OVER()` Clause, Ranking Algorithms (`ROW_NUMBER` vs `RANK` vs `DENSE_RANK`), Partitions (`PARTITION BY`), Top-N Per Group Filtering  
**Status:** Completed & Pre-Executed  

---

### What are Window Functions?
Unlike regular aggregate functions (`SUM`, `AVG`, `COUNT` with `GROUP BY`) which collapse multiple rows into a single summary row, **Window Functions** compute calculations across a set of table rows related to the current row **without collapsing the rows**. Every individual row retains its separate identity in the final output.

```text
+-------------------+-----------------------------------------------------------------------------------------+
| Window Function   | Behavior on Ties                                                                        |
+-------------------+-----------------------------------------------------------------------------------------+
| ROW_NUMBER()      | Always produces strict unique sequential integers (1, 2, 3, 4...). Never repeats.       |
| RANK()            | Assigns same rank to tied rows, then skips numbers (e.g. 1, 2, 2, 4). Gaps exist.       |
| DENSE_RANK()      | Assigns same rank to tied rows, but does NOT skip numbers (e.g. 1, 2, 2, 3). No gaps.  |
+-------------------+-----------------------------------------------------------------------------------------+
```


## Task 1: Create 'Playlists' Table & Populate Sample Data
> **Problem Statement:** Create a table named `Playlists` with columns: `id`, `user_id`, `playlist_name`, and `total_likes`. Insert at least 8 sample rows with different users and playlists, making sure some playlists have the same `user_id`.

### Schema Architecture
- Entity representing public Spotify playlists created by users.
- Includes realistic playlists and likes metrics across multiple creators.
- **Ties included:** Multiple playlists have identical like counts (e.g., 15,000 and 12,000) to explicitly highlight the behavior of `ROW_NUMBER()`, `RANK()`, and `DENSE_RANK()`.


In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('spotify_windows.db')
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS Playlists;")
cursor.execute("""
CREATE TABLE Playlists (
    id INT PRIMARY KEY,
    user_id INT NOT NULL,
    playlist_name VARCHAR(100) NOT NULL,
    total_likes INT NOT NULL
);
""")

playlists_data = [
    # User 101 (Atharv)
    (1, 101, 'Gym Workout Beats', 15000),
    (2, 101, 'Coding Lo-Fi Focus', 12000),
    (3, 101, 'Rock Classics', 12000),         # Tied with Coding Lo-Fi Focus
    (4, 101, 'Chill Acoustic', 8500),
    # User 102 (Sneha)
    (5, 102, 'Bollywood Dance Party', 18500),
    (6, 102, 'Road Trip Anthems', 15000),      # Tied with Gym Workout Beats
    (7, 102, 'Evening Jazz & Chai', 9200),
    # User 103 (Rahul)
    (8, 103, 'Deep House Vibes', 14000),
    (9, 103, 'EDM Festival Bangers', 12000),   # Tied with other 12000 playlists
    (10, 103, 'Morning Motivation', 6000)
]
cursor.executemany("INSERT INTO Playlists VALUES (?, ?, ?, ?);", playlists_data)
conn.commit()

df_raw = pd.read_sql_query("SELECT * FROM Playlists;", conn)
display(df_raw)


,id,user_id,playlist_name,total_likes
0,1,101,Gym Workout Beats,15000
1,2,101,Coding Lo-Fi Focus,12000
2,3,101,Rock Classics,12000
3,4,101,Chill Acoustic,8500
4,5,102,Bollywood Dance Party,18500
5,6,102,Road Trip Anthems,15000
6,7,102,Evening Jazz & Chai,9200
7,8,103,Deep House Vibes,14000
8,9,103,EDM Festival Bangers,12000
9,10,103,Morning Motivation,6000


## Task 2: ROW_NUMBER() with OVER() Clause
> **Problem Statement:** Write a SQL query using `ROW_NUMBER()` and the `OVER()` clause to assign a unique row number to each playlist, ordered by `total_likes` in descending order.

### Conceptual Explanation
`ROW_NUMBER()` assigns a strictly unique, contiguous integer sequence starting from `1` up to the total count of rows:
```sql
ROW_NUMBER() OVER (ORDER BY total_likes DESC)
```
Even if two playlists have identical like counts (e.g., *Gym Workout Beats* and *Road Trip Anthems* both have 15,000 likes), `ROW_NUMBER()` arbitrarily breaks the tie to ensure no numbers are repeated. This makes it ideal for web pagination and deduplication.


In [2]:
task2_query = """
SELECT 
    ROW_NUMBER() OVER (ORDER BY total_likes DESC) AS row_num,
    id AS playlist_id,
    playlist_name,
    user_id,
    total_likes
FROM Playlists;
"""

df_t2 = pd.read_sql_query(task2_query, conn)
display(df_t2)


,row_num,playlist_id,playlist_name,user_id,total_likes
0,1,5,Bollywood Dance Party,102,18500
1,2,1,Gym Workout Beats,101,15000
2,3,6,Road Trip Anthems,102,15000
3,4,8,Deep House Vibes,103,14000
4,5,2,Coding Lo-Fi Focus,101,12000
5,6,3,Rock Classics,101,12000
6,7,9,EDM Festival Bangers,103,12000
7,8,7,Evening Jazz & Chai,102,9200
8,9,4,Chill Acoustic,101,8500
9,10,10,Morning Motivation,103,6000


## Task 3: RANK() with OVER() Clause
> **Problem Statement:** Use the `RANK()` function with the `OVER()` clause to rank all playlists by `total_likes`, and display the `playlist_name`, `user_id`, `total_likes`, and their rank.

### Conceptual Explanation
`RANK()` implements standard competition ranking (like Olympic medals):
```sql
RANK() OVER (ORDER BY total_likes DESC)
```
- When items share the exact same value, they share the **same rank**.
- The next subsequent rank is incremented by the number of tied items, creating **gaps** in the sequence.
- *Example:* Both playlists with 15,000 likes share Rank **2**. The next playlist receives Rank **4** (Rank 3 is skipped).


In [3]:
task3_query = """
SELECT 
    RANK() OVER (ORDER BY total_likes DESC) AS rank_pos,
    playlist_name,
    user_id,
    total_likes
FROM Playlists;
"""

df_t3 = pd.read_sql_query(task3_query, conn)
display(df_t3)


,rank_pos,playlist_name,user_id,total_likes
0,1,Bollywood Dance Party,102,18500
1,2,Gym Workout Beats,101,15000
2,2,Road Trip Anthems,102,15000
3,4,Deep House Vibes,103,14000
4,5,Coding Lo-Fi Focus,101,12000
5,5,Rock Classics,101,12000
6,5,EDM Festival Bangers,103,12000
7,8,Evening Jazz & Chai,102,9200
8,9,Chill Acoustic,101,8500
9,10,Morning Motivation,103,6000


## Task 4: DENSE_RANK() with PARTITION BY user_id
> **Problem Statement:** Write a SQL query using `DENSE_RANK()` and `PARTITION BY user_id` to rank each user's playlists by `total_likes`, showing `playlist_name`, `user_id`, `total_likes`, and dense rank.  
> *Hint: This will show how popular each playlist is within each user's account, similar to how Spotify might rank your top playlists.*

### Conceptual Explanation
1. **`PARTITION BY user_id`:** Splits the execution window into independent partitions per user. The ranking restarts at `1` whenever `user_id` changes.
2. **`DENSE_RANK()`:** Ties receive the same rank number, but **no ranks are skipped** (e.g. 1, 2, 2, 3).


In [4]:
task4_query = """
SELECT 
    user_id,
    DENSE_RANK() OVER (
        PARTITION BY user_id 
        ORDER BY total_likes DESC
    ) AS user_dense_rank,
    playlist_name,
    total_likes
FROM Playlists
ORDER BY user_id ASC, user_dense_rank ASC;
"""

df_t4 = pd.read_sql_query(task4_query, conn)
display(df_t4)


,user_id,user_dense_rank,playlist_name,total_likes
0,101,1,Gym Workout Beats,15000
1,101,2,Coding Lo-Fi Focus,12000
2,101,2,Rock Classics,12000
3,101,3,Chill Acoustic,8500
4,102,1,Bollywood Dance Party,18500
5,102,2,Road Trip Anthems,15000
6,102,3,Evening Jazz & Chai,9200
7,103,1,Deep House Vibes,14000
8,103,2,EDM Festival Bangers,12000
9,103,3,Morning Motivation,6000


## Task 5: Top 2 Playlists Per User (Spotify 'Your Top Playlists')
> **Problem Statement:** Imagine you want to show the top 2 playlists per user based on `total_likes`, like Spotify's 'Your Top Playlists' feature. Write a query using a window function to select only the top 2 playlists for each user.

---

### Engineering Pattern: Top-N Per Group
SQL does not permit filtering window functions directly in the `WHERE` clause of the same query block (because window functions are evaluated in Step 6 of logical query processing, **after** `WHERE`).

To solve this, we wrap the window calculation inside a **CTE**, and then apply `WHERE rank_in_user <= 2` in the outer query:
```sql
WITH RankedUserPlaylists AS (
    SELECT 
        user_id,
        playlist_name,
        total_likes,
        ROW_NUMBER() OVER (
            PARTITION BY user_id 
            ORDER BY total_likes DESC
        ) AS rank_in_user
    FROM Playlists
)
SELECT * FROM RankedUserPlaylists
WHERE rank_in_user <= 2;
```


In [5]:
task5_query = """
WITH RankedUserPlaylists AS (
    SELECT 
        user_id,
        playlist_name,
        total_likes,
        ROW_NUMBER() OVER (
            PARTITION BY user_id 
            ORDER BY total_likes DESC
        ) AS rank_in_user
    FROM Playlists
)
SELECT 
    user_id,
    rank_in_user AS playlist_rank,
    playlist_name,
    total_likes
FROM RankedUserPlaylists
WHERE rank_in_user <= 2
ORDER BY user_id ASC, rank_in_user ASC;
"""

df_t5 = pd.read_sql_query(task5_query, conn)
display(df_t5)

conn.close()


,user_id,playlist_rank,playlist_name,total_likes
0,101,1,Gym Workout Beats,15000
1,101,2,Coding Lo-Fi Focus,12000
2,102,1,Bollywood Dance Party,18500
3,102,2,Road Trip Anthems,15000
4,103,1,Deep House Vibes,14000
5,103,2,EDM Festival Bangers,12000


## Summary & Ranking Cheat Sheet

| Feature | `ROW_NUMBER()` | `RANK()` | `DENSE_RANK()` |
| :--- | :--- | :--- | :--- |
| **Tie Handling** | Assigns distinct arbitrary numbers | Same rank to ties | Same rank to ties |
| **Gaps in Numbers?** | No gaps (1, 2, 3, 4, 5) | Yes, skips ranks (1, 2, 2, 4) | No gaps (1, 2, 2, 3) |
| **Best Used For** | Pagination, unique row IDs, deduplication | Competition leaderboards, official standings | Tiered brackets, top percentiles, grouped ranks |
| **With `PARTITION BY`** | Resets sequence per partition | Resets rankings per partition | Resets dense ranks per partition |
